# Air Quality PM2.5 — SageMaker Training Notebook

Run the full ML pipeline from a SageMaker Unified Studio notebook:
1. **Data exploration** — inspect raw data in S3
2. **Feature engineering** — build training features
3. **Model training** — train XGBoost on SageMaker
4. **Evaluation** — review metrics and feature importance
5. **Model registry** — register approved model

## 0. Setup

In [ ]:
# Install dependencies (run once)
!pip install -q xgboost pandas numpy scikit-learn pyarrow boto3 matplotlib seaborn loguru

In [ ]:
import json
import os
import tarfile
import tempfile
from datetime import datetime, timezone
from io import BytesIO

import boto3
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import xgboost as xgb
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

sns.set_style("whitegrid")
pd.set_option("display.max_columns", 20)

In [ ]:
# Configuration — update these for your environment
S3_BUCKET = "air-quality-mlops-data"
REGION = "eu-north-1"
SAGEMAKER_ROLE_ARN = "arn:aws:iam::735234196998:role/service-role/AmazonSageMaker-ExecutionRole-20260305T200038"
TRAINING_INSTANCE_TYPE = "ml.m5.xlarge"
MODEL_PACKAGE_GROUP = "AirQualityPM25Model"
RMSE_THRESHOLD = 15.0

# SKLearn container (supports Script Mode — runs our train_model.py)
SKLEARN_IMAGE_URI = "662702820516.dkr.ecr.eu-north-1.amazonaws.com/sagemaker-scikit-learn:1.2-1-cpu-py3"

s3 = boto3.client("s3", region_name=REGION)
sm = boto3.client("sagemaker", region_name=REGION)

print(f"Bucket: {S3_BUCKET}")
print(f"Region: {REGION}")
print(f"Role: {SAGEMAKER_ROLE_ARN}")

## 1. Data Exploration

In [ ]:
# List raw data files in S3
response = s3.list_objects_v2(Bucket=S3_BUCKET, Prefix="raw/")
raw_files = [obj["Key"] for obj in response.get("Contents", []) if obj["Key"].endswith(".parquet")]
print(f"Found {len(raw_files)} raw parquet files")
for f in raw_files[:10]:
    print(f"  {f}")

In [ ]:
# Load all raw data
dfs = []
for key in raw_files:
    obj = s3.get_object(Bucket=S3_BUCKET, Key=key)
    dfs.append(pd.read_parquet(BytesIO(obj["Body"].read())))

raw_df = pd.concat(dfs, ignore_index=True)
raw_df["timestamp"] = pd.to_datetime(raw_df["timestamp"])
print(f"Total raw rows: {len(raw_df)}")
raw_df.head()

In [ ]:
raw_df.describe()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].hist(raw_df["pm25"], bins=50, edgecolor="black")
axes[0, 0].set_title("PM2.5 Distribution")
axes[0, 0].set_xlabel("PM2.5 (µg/m³)")

axes[0, 1].hist(raw_df["temperature"], bins=50, edgecolor="black", color="orange")
axes[0, 1].set_title("Temperature Distribution")
axes[0, 1].set_xlabel("Temperature (°C)")

axes[1, 0].hist(raw_df["humidity"], bins=50, edgecolor="black", color="green")
axes[1, 0].set_title("Humidity Distribution")
axes[1, 0].set_xlabel("Relative Humidity (%)")

axes[1, 1].hist(raw_df["wind_speed"], bins=50, edgecolor="black", color="purple")
axes[1, 1].set_title("Wind Speed Distribution")
axes[1, 1].set_xlabel("Wind Speed (m/s)")

plt.tight_layout()
plt.show()

In [ ]:
# Correlation matrix
numeric_cols = ["pm25", "temperature", "humidity", "wind_speed"]
corr = raw_df[numeric_cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, cmap="coolwarm", center=0, fmt=".2f")
plt.title("Feature Correlations")
plt.show()

In [ ]:
# PM2.5 over time (daily average)
daily = raw_df.set_index("timestamp").resample("D")["pm25"].mean()

plt.figure(figsize=(14, 4))
daily.plot()
plt.title("Daily Average PM2.5 — Beijing")
plt.ylabel("PM2.5 (µg/m³)")
plt.xlabel("Date")
plt.tight_layout()
plt.show()

## 2. Feature Engineering

In [ ]:
# Build features with PM2.5 lag features (key predictors)
df = raw_df.sort_values(["latitude", "longitude", "timestamp"]).reset_index(drop=True)
grouped = df.groupby(["latitude", "longitude"])

# Time features
df["hour"] = df["timestamp"].dt.hour
df["day_of_week"] = df["timestamp"].dt.dayofweek
df["is_rush_hour"] = df["hour"].isin([7, 8, 9, 17, 18, 19]).astype(int)

# PM2.5 lag features — strongest predictors
df["pm25_lag_1h"] = grouped["pm25"].transform(lambda x: x.shift(1))
df["pm25_lag_3h"] = grouped["pm25"].transform(lambda x: x.shift(3))
df["pm25_rolling_mean_3h"] = grouped["pm25"].transform(
    lambda x: x.rolling(3, min_periods=1).mean()
)

# Target: next hour's PM2.5
df["pm25_target"] = grouped["pm25"].transform(lambda x: x.shift(-1))

# Select features
feature_cols = ["latitude", "longitude", "temperature", "humidity", "wind_speed",
                "hour", "day_of_week", "is_rush_hour",
                "pm25_lag_1h", "pm25_lag_3h", "pm25_rolling_mean_3h"]
output_cols = ["timestamp"] + feature_cols + ["pm25_target"]
features_df = df[output_cols].dropna().reset_index(drop=True)

print(f"Feature dataset: {features_df.shape}")
print(f"Features: {feature_cols}")
features_df.head()

In [ ]:
# Upload feature dataset to S3
buf = BytesIO()
features_df.to_parquet(buf, index=False)
buf.seek(0)
s3.put_object(Bucket=S3_BUCKET, Key="features/training_dataset.parquet", Body=buf.getvalue())
print(f"Uploaded {len(features_df)} rows to s3://{S3_BUCKET}/features/training_dataset.parquet")

## 3. Local Training (quick validation before SageMaker)

In [ ]:
X = features_df[feature_cols]
y = features_df["pm25_target"]

# Time-based split (last 20% for test)
split_idx = int(len(features_df) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

print(f"Train: {len(X_train)} rows, Test: {len(X_test)} rows")

In [ ]:
model = xgb.XGBRegressor(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
)
model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=20)

In [ ]:
y_pred = model.predict(X_test)
rmse = float(np.sqrt(mean_squared_error(y_test, y_pred)))
mae = float(mean_absolute_error(y_test, y_pred))
r2 = float(r2_score(y_test, y_pred))

print(f"RMSE: {rmse:.2f}")
print(f"MAE:  {mae:.2f}")
print(f"R²:   {r2:.3f}")
print(f"\nAuto-approval threshold: RMSE < {RMSE_THRESHOLD} → {'APPROVED' if rmse < RMSE_THRESHOLD else 'PENDING'}")

In [ ]:
# Feature importance
importance = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=True)

plt.figure(figsize=(10, 5))
importance.plot.barh()
plt.title("Feature Importance")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

In [ ]:
# Actual vs Predicted
plt.figure(figsize=(10, 5))
plt.scatter(y_test, y_pred, alpha=0.3, s=10)
plt.plot([0, y_test.max()], [0, y_test.max()], "r--", lw=2)
plt.xlabel("Actual PM2.5")
plt.ylabel("Predicted PM2.5")
plt.title(f"Actual vs Predicted (RMSE={rmse:.2f})")
plt.tight_layout()
plt.show()

In [ ]:
# Residuals over time
residuals = y_test.values - y_pred
test_timestamps = features_df["timestamp"].iloc[split_idx:].values

plt.figure(figsize=(14, 4))
plt.scatter(test_timestamps, residuals, alpha=0.2, s=5)
plt.axhline(y=0, color="r", linestyle="--")
plt.title("Prediction Residuals Over Time")
plt.ylabel("Residual (actual - predicted)")
plt.xlabel("Time")
plt.tight_layout()
plt.show()

## 4. Package and Upload Model to S3

Save the locally trained model as `model.tar.gz` so the API can use it immediately.

In [ ]:
# Save model artifact
tmp_dir = tempfile.mkdtemp()
model_path = os.path.join(tmp_dir, "xgboost-model.json")
model.save_model(model_path)

metrics = {"rmse": rmse, "mae": mae, "r2": r2}
with open(os.path.join(tmp_dir, "metrics.json"), "w") as f:
    json.dump(metrics, f, indent=2)

with open(os.path.join(tmp_dir, "feature_names.json"), "w") as f:
    json.dump(feature_cols, f)

# Create tar.gz
tar_path = os.path.join(tmp_dir, "model.tar.gz")
with tarfile.open(tar_path, "w:gz") as tar:
    tar.add(os.path.join(tmp_dir, "xgboost-model.json"), arcname="xgboost-model.json")
    tar.add(os.path.join(tmp_dir, "metrics.json"), arcname="metrics.json")
    tar.add(os.path.join(tmp_dir, "feature_names.json"), arcname="feature_names.json")

# Upload to S3
s3.upload_file(tar_path, S3_BUCKET, "models/model.tar.gz")
print(f"Uploaded model.tar.gz to s3://{S3_BUCKET}/models/model.tar.gz")

# Upload metrics separately
s3.put_object(
    Bucket=S3_BUCKET,
    Key="models/latest_metrics.json",
    Body=json.dumps(metrics, indent=2),
)
print(f"Uploaded metrics: {metrics}")

## 5. Launch SageMaker Training Job (Script Mode)

This packages our `train_model.py` and runs it on a SageMaker instance. The script:
- Loads parquet data directly
- Trains XGBoost with scikit-learn API
- Saves model as JSON (compatible with any XGBoost version)
- Saves `metrics.json` and `feature_names.json`

In [ ]:
import io

timestamp = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")
job_name = f"air-quality-pm25-{timestamp}"

# Package train_model.py as source tarball
script_content = open("../training/train_model.py", "rb").read()
source_buf = io.BytesIO()
with tarfile.open(fileobj=source_buf, mode="w:gz") as tar:
    info = tarfile.TarInfo(name="train_model.py")
    info.size = len(script_content)
    tar.addfile(info, io.BytesIO(script_content))
source_buf.seek(0)

s3.put_object(Bucket=S3_BUCKET, Key="training-scripts/sourcedir.tar.gz", Body=source_buf.getvalue())
source_dir_uri = f"s3://{S3_BUCKET}/training-scripts/sourcedir.tar.gz"

print(f"Job name:    {job_name}")
print(f"Image:       {SKLEARN_IMAGE_URI}")
print(f"Instance:    {TRAINING_INSTANCE_TYPE}")
print(f"Script:      {source_dir_uri}")

In [ ]:
# Launch the training job (Script Mode — runs our train_model.py)
sm.create_training_job(
    TrainingJobName=job_name,
    AlgorithmSpecification={
        "TrainingImage": SKLEARN_IMAGE_URI,
        "TrainingInputMode": "File",
    },
    RoleArn=SAGEMAKER_ROLE_ARN,
    InputDataConfig=[
        {
            "ChannelName": "training",
            "DataSource": {
                "S3DataSource": {
                    "S3DataType": "S3Prefix",
                    "S3Uri": f"s3://{S3_BUCKET}/features/",
                    "S3DataDistributionType": "FullyReplicated",
                }
            },
            "ContentType": "application/x-parquet",
        }
    ],
    OutputDataConfig={
        "S3OutputPath": f"s3://{S3_BUCKET}/models/sagemaker/",
    },
    ResourceConfig={
        "InstanceCount": 1,
        "InstanceType": TRAINING_INSTANCE_TYPE,
        "VolumeSizeInGB": 10,
    },
    StoppingCondition={"MaxRuntimeInSeconds": 3600},
    HyperParameters={
        "sagemaker_program": "train_model.py",
        "sagemaker_submit_directory": source_dir_uri,
        "n-estimators": "200",
        "max-depth": "6",
        "learning-rate": "0.1",
        "subsample": "0.8",
        "colsample-bytree": "0.8",
    },
)

print(f"Training job '{job_name}' launched!")

In [ ]:
# Wait for training job to complete
import time

print("Waiting for training job to complete...")
while True:
    response = sm.describe_training_job(TrainingJobName=job_name)
    status = response["TrainingJobStatus"]
    print(f"  Status: {status}", end="")
    
    if status in ("Completed", "Failed", "Stopped"):
        print()
        break
    
    secondary = response.get("SecondaryStatus", "")
    print(f" ({secondary})")
    time.sleep(30)

if status == "Completed":
    model_artifact = response["ModelArtifacts"]["S3ModelArtifacts"]
    print(f"\nTraining complete!")
    print(f"Model artifact: {model_artifact}")
else:
    failure = response.get("FailureReason", "Unknown")
    print(f"\nTraining FAILED: {status} — {failure}")

In [ ]:
# Copy SageMaker model to standard location and extract metrics
if status == "Completed":
    source_key = model_artifact.replace(f"s3://{S3_BUCKET}/", "")
    s3.copy_object(
        Bucket=S3_BUCKET,
        CopySource={"Bucket": S3_BUCKET, "Key": source_key},
        Key="models/model.tar.gz",
    )
    print(f"Copied model to s3://{S3_BUCKET}/models/model.tar.gz")
    
    # Extract metrics from model artifact (our script saves metrics.json)
    tar_path = os.path.join(tempfile.gettempdir(), "sm_model.tar.gz")
    s3.download_file(S3_BUCKET, "models/model.tar.gz", tar_path)
    sm_metrics = {}
    with tarfile.open(tar_path, "r:gz") as tar:
        # List contents
        print(f"Model artifact contents: {tar.getnames()}")
        try:
            f = tar.extractfile("metrics.json")
            if f:
                sm_metrics = json.loads(f.read())
                s3.put_object(
                    Bucket=S3_BUCKET,
                    Key="models/latest_metrics.json",
                    Body=json.dumps(sm_metrics, indent=2),
                )
        except KeyError:
            print("No metrics.json found")
    
    print(f"Metrics: {sm_metrics}")

## 6. Register Model in SageMaker Model Registry

In [ ]:
# Ensure model package group exists
try:
    sm.create_model_package_group(
        ModelPackageGroupName=MODEL_PACKAGE_GROUP,
        ModelPackageGroupDescription="Air Quality PM2.5 prediction models for Beijing",
    )
    print(f"Created model package group: {MODEL_PACKAGE_GROUP}")
except sm.exceptions.ClientError as e:
    if "already exists" in str(e).lower():
        print(f"Model package group '{MODEL_PACKAGE_GROUP}' already exists")
    else:
        raise

In [ ]:
# Read latest metrics for approval decision
try:
    resp = s3.get_object(Bucket=S3_BUCKET, Key="models/latest_metrics.json")
    reg_metrics = json.loads(resp["Body"].read())
except Exception:
    reg_metrics = {}

reg_rmse = reg_metrics.get("rmse", float("inf"))
approval_status = "Approved" if reg_rmse < RMSE_THRESHOLD else "PendingManualApproval"

print(f"RMSE: {reg_rmse:.2f}, Threshold: {RMSE_THRESHOLD}")
print(f"Approval status: {approval_status}")

In [ ]:
# Register model
model_data_url = f"s3://{S3_BUCKET}/models/model.tar.gz"

response = sm.create_model_package(
    ModelPackageGroupName=MODEL_PACKAGE_GROUP,
    InferenceSpecification={
        "Containers": [
            {
                "Image": SKLEARN_IMAGE_URI,
                "ModelDataUrl": model_data_url,
            }
        ],
        "SupportedContentTypes": ["text/csv"],
        "SupportedResponseMIMETypes": ["text/csv"],
    },
    ModelApprovalStatus=approval_status,
    ModelPackageDescription=f"RMSE: {reg_rmse:.2f}" if reg_metrics else "No metrics available",
)

model_package_arn = response["ModelPackageArn"]
print(f"\nRegistered model: {model_package_arn}")
print(f"Status: {approval_status}")

In [ ]:
# List all registered model versions
versions = sm.list_model_packages(
    ModelPackageGroupName=MODEL_PACKAGE_GROUP,
    SortBy="CreationTime",
    SortOrder="Descending",
)

print(f"\nModel versions in '{MODEL_PACKAGE_GROUP}':")
for pkg in versions["ModelPackageSummaryList"]:
    print(f"  {pkg['ModelPackageArn'].split('/')[-1]:>3} | "
          f"{pkg['ModelApprovalStatus']:<25} | "
          f"{pkg['CreationTime'].strftime('%Y-%m-%d %H:%M')}")

## 7. Summary

In [ ]:
# Read final metrics
try:
    resp = s3.get_object(Bucket=S3_BUCKET, Key="models/latest_metrics.json")
    summary_metrics = json.loads(resp["Body"].read())
except Exception:
    summary_metrics = {}

print("=" * 60)
print("  Air Quality PM2.5 — Training Summary")
print("=" * 60)
print(f"  Training rows:    {len(features_df)}")
print(f"  Features:         {len(feature_cols)}")
print(f"  RMSE:             {summary_metrics.get('rmse', 'N/A')}")
print(f"  Model artifact:   s3://{S3_BUCKET}/models/model.tar.gz")
print(f"  Registry:         {MODEL_PACKAGE_GROUP}")
print(f"  Approval:         {approval_status}")
print("=" * 60)